# Can concepts predict temporal performance deterioration?

At the end of year **t**, we know the fitted model's concept measurements and observed performance through **t**. We predict the change to **t+1**. Positive deterioration means worse performance: current minus next for frozen-cutoff F1 and average precision; next minus current for Brier score.

The mortality cutoff is chosen on reference-year validation patients and stays fixed. Oracle cutoffs are descriptive only. Brier score measures overall probability error, not calibration alone.

This notebook only reads validated, completed artifacts. Run `run_temporal_concept_forecasting.py` to produce them; executing this notebook cannot launch experiments.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from artifact_storage import file_sha256, read_artifact
from temporal_concept_forecasting import checked_manifest, load_forecasting

if 'REPORT_MANIFEST' not in globals():
    pointer_path = Path('stats/temporal_concept_forecasting/report_manifest.json')
    if not pointer_path.exists():
        raise FileNotFoundError('No report artifacts yet. Run run_temporal_concept_forecasting.py first.')
    pointer = json.loads(pointer_path.read_text())
    REPORT_MANIFEST = pointer['path']
    assert file_sha256(REPORT_MANIFEST) == pointer['sha256']
report_path = Path(REPORT_MANIFEST)
report = checked_manifest(report_path)
runs = {}
for source in report['forecast_manifests']:
    assert file_sha256(source['path']) == source['sha256']
    data = load_forecasting(source['path'])
    runs[data['manifest']['system']] = data
print('Systems:', ', '.join(runs))
print('Stage B complete:', report['stage_b_complete'])
if not report['stage_b_complete']:
    display(Markdown('**Training-window analysis is not complete. No conclusion about its concept predictability is available yet.**'))
comparisons = pd.DataFrame([r for data in runs.values() for r in data['comparisons']])
folds = pd.DataFrame([r for data in runs.values() for r in data['folds']])
performance = pd.DataFrame([r for data in runs.values() for r in data['performance']])

## Primary result

Does history plus the five concept measurements forecast frozen-cutoff F1 deterioration more accurately than history alone? The primary population is all-comers, activation fraction 0.5, with forward evaluation.

An improvement above zero means lower forecasting error. Intervals resample whole reference years, after averaging split losses within forecast years. References reuse calendar years and some patients; these intervals do not establish generalization to independent populations.

In [ ]:
primary = comparisons[comparisons['primary'] == True] if not comparisons.empty else pd.DataFrame()
if primary.empty:
    print('Primary comparison is not estimable; inspect exclusions below.')
else:
    display(primary[['system','improvement','lower_95','upper_95','paired_rows','references']])
    for r in primary.to_dict('records'):
        supported = r['lower_95'] is not None and pd.notna(r['lower_95']) and r['lower_95'] > 0
        print(f"{r['system']}: {'evidence of lower forecasting error' if supported else 'no clear improvement established'} on {r['references']} held reference years.")

## Other outcomes and representations

Each outcome, cohort, activation fraction and representation is evaluated separately. History includes current performance, its previous change, and model age. Concepts add current values, previous-year changes and coverage. PCA, factor analysis, CRI normalization, scaling and DAE are fitted inside the training folds. DAE uses the prespecified architecture and 250 epochs, averaging predictions across seeds 42–44.

The TCAV extension is secondary; its missing measurements do not remove rows from the core analysis. Each comparison below uses identical forecast rows for the two models. History with maximal coverage is reported separately.

In [ ]:
if not comparisons.empty:
    selected = comparisons.query("design == 'forward' and cohort_view == 'all_comer' and activation_target == 0.5 and profile == 'core' and metric == 'mae'")
    display(selected[['system','outcome','model','baseline','improvement','lower_95','upper_95','paired_rows','references']])
    fig, ax = plt.subplots(figsize=(9, max(3, len(primary)*0.7)))
    if not primary.empty:
        p = primary.reset_index(drop=True)
        ax.axvline(0, color='gray', linewidth=1)
        ax.scatter(p['improvement'], range(len(p)))
        for i, r in p.iterrows():
            if pd.notna(r['lower_95']) and pd.notna(r['upper_95']):
                ax.plot([r['lower_95'], r['upper_95']], [i,i], color='C0')
        ax.set_yticks(range(len(p)), p['system'])
        ax.set_xlabel('Reduction in forecasting MAE versus history (positive is better)')
        ax.set_title('Primary forward comparison with 95% reference-cluster intervals')
    plt.tight_layout()
    plt.show()

## Coverage and timing audit

Missing concepts are exclusions, not zero robustness. Early forward folds may have too few earlier reference systems to fit a forecasting model. Inspect support before interpreting differences.

In [ ]:
if not folds.empty:
    audit = folds[folds['status'].isin(['non_estimable','excluded'])]
    display(audit.groupby(['system','status','reason'], dropna=False).size().rename('fold/model entries').reset_index())
    forward = folds[folds['design']=='forward']
    assert all(all(y < r['held_reference'] for y in r['train_references']) for r in forward.to_dict('records'))
    assert all(pd.isna(r['maximum_training_outcome_year']) or r['maximum_training_outcome_year'] <= r['origin'] for r in forward.to_dict('records'))
    print('Forward timing checks passed.')
summary = pd.DataFrame([r for data in runs.values() for r in data['summaries']])
if not summary.empty:
    display(summary.query("model in ['history','history_maximal','no_change'] and activation_target == 0.5 and profile == 'core'"))

## Training windows: three separate questions

1. Are performance levels better?
2. Is subsequent deterioration smaller?
3. Do concepts forecast deterioration more accurately beyond history?

These are different claims. Comparisons use common reference-only as the baseline and require identical record membership. Factor identities stay local to each fitted system. Expanding training and concept discovery together does not identify the separate contribution of either change.

In [ ]:
window_comparisons = pd.DataFrame(read_artifact(report_path.parent, report['artifacts']['window_comparisons']))
if window_comparisons.empty:
    print('Matched training-window comparisons are not available.')
else:
    display(window_comparisons[window_comparisons['cohort_view']=='all_comer'])
    print('All differences are window minus common reference-only.')
    print('Performance: higher F1/AP and lower Brier are better. Deterioration: lower is better. Concept forecasting gain: higher is better.')
if not performance.empty:
    all_comer = performance[(performance['cohort_view']=='all_comer') & (performance['valid']==True)]
    trajectory = all_comer.groupby(['system','reference_year','temporal_distance'])['death_f1_at_frozen_threshold'].mean().groupby(['system','temporal_distance']).mean().unstack(0)
    trajectory.plot(marker='o', ylabel='Frozen-cutoff death F1', xlabel='Years since reference', title='Descriptive performance trajectories; reference support varies by distance', figsize=(9,4))
    plt.tight_layout()
    plt.show()